In [6]:
import cv2
import os

# Tạo thư mục nếu chưa có
true_pose_dir = 'trainAI/true_pose/'     #vị trí lưu hình ngồi đúng tư thế
false_pose_dir = 'trainAI/false_pose/'   #vị trí lưu hình ngồi sai tư thế


os.makedirs(true_pose_dir, exist_ok=True)
os.makedirs(false_pose_dir, exist_ok=True)


cap = cv2.VideoCapture(0)


# Kiểm tra xem camera có mở thành công không
if not cap or not cap.isOpened():
    print("Không thể mở camera. Vui lòng kiểm tra kết nối webcam hoặc quyền truy cập.")
else:
    # Nhập tên của các lớp tư thế
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Không thể nhận khung hình từ camera. Thoát.")
            break

        # Hiển thị ảnh
        cv2.imshow('Camera', frame)
        key = cv2.waitKey(1) & 0xFF

        # Nếu nhấn phím 1, lưu ảnh vào thư mục true_pose
        if key == ord('1'):
            img_name = f"{true_pose_dir}true_{str(len(os.listdir(true_pose_dir)))}.jpg"
            cv2.imwrite(img_name, frame)
            print(f"Saved true pose: {img_name}")

        # Nếu nhấn phím 2, lưu ảnh vào thư mục false_pose
        elif key == ord('2'):
            img_name = f"{false_pose_dir}false_{str(len(os.listdir(false_pose_dir)))}.jpg"
            cv2.imwrite(img_name, frame)
            print(f"Saved false pose: {img_name}")

        # Thoát khi nhấn phím Q
        elif key == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [11]:
import cv2
import tensorflow as tf
import mediapipe as mp
import numpy as np

# Tải mô hình đã huấn luyện
# Make sure the model file is in the correct path
# If your model is in a different location, update the path below
model = tf.keras.models.load_model('mymodel.h5') # Update this path if needed

# Khởi tạo MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

# Khởi tạo webcam
cap = cv2.VideoCapture(0) # Use 0 for the default webcam

if not cap.isOpened():
    print("Không thể mở camera. Vui lòng kiểm tra kết nối webcam.")
else:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Không thể nhận khung hình từ camera. Thoát.")
            break

        # Chuyển ảnh thành RGB
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Phát hiện các điểm mốc tư thế
        results = pose.process(image_rgb)

        # Vẽ các điểm mốc và kết nối
        if results.pose_landmarks:
            mp.solutions.drawing_utils.draw_landmarks(
                frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                mp.solutions.drawing_utils.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                mp.solutions.drawing_utils.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
            )

            # Trích xuất các điểm mốc cho dự đoán
            landmarks = results.pose_landmarks
            points = []
            for i in range(13):  # Lấy 13 điểm mốc đầu (từ 0 đến 12)
                x = landmarks.landmark[i].x
                y = landmarks.landmark[i].y
                z = landmarks.landmark[i].z
                visibility = landmarks.landmark[i].visibility
                points.extend([x, y, z, visibility])

            # Tiền xử lý dữ liệu đầu vào cho mô hình
            input_data = np.array(points).reshape(1, -1).astype('float32')

            # Dự đoán từ mô hình
            prediction = model.predict(input_data, verbose=0)
            prediction_label = 'True Pose' if prediction[0] > 0.5 else 'False Pose'

            # Hiển thị kết quả lên ảnh
            cv2.putText(frame, prediction_label, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # Hiển thị webcam
        cv2.imshow('Webcam Pose Detection', frame)

        # Thoát khi nhấn phím 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Clean up
    cap.release()
    cv2.destroyAllWindows()

KeyboardInterrupt: 

In [ ]:
import torch
import cv2
import numpy as np
# Import lớp YOLO từ thư viện ultralytics
from ultralytics import YOLO  # Đảm bảo bạn đã cài đặt: pip install ultralytics

# Tải mô hình YOLO Pose
# Sử dụng lớp YOLO để tải file .pt/model
model = YOLO('yolo11n-pose.pt') 

# Khởi tạo webcam
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # *** Dự đoán sử dụng phương thức 'predict' của đối tượng YOLO ***
    # Sử dụng frame (ảnh OpenCV) trực tiếp
    results = model.predict(source=frame, device='cpu', verbose=False)
    
    # Lấy kết quả cho frame hiện tại (thường chỉ có 1 kết quả)
    if results:
        r = results[0]  # Lấy kết quả đầu tiên
        keypoints_data = r.keypoints.xyn  # Lấy tọa độ (x, y) chuẩn hóa (0-1)
        
        # Nếu có keypoints được phát hiện
        if keypoints_data is not None:
            keypoints_numpy = keypoints_data.cpu().numpy()[0] # Lấy keypoints của người đầu tiên

            # Lấy tọa độ các điểm mốc và vẽ chúng lên ảnh
            for i in range(keypoints_numpy.shape[0]):  # Duyệt qua từng điểm mốc (ví dụ: 17 điểm)
                x_norm, y_norm = keypoints_numpy[i] 
                
                # Quy đổi từ tỉ lệ ra pixel
                cx, cy = int(x_norm * frame.shape[1]), int(y_norm * frame.shape[0])  
                
                # Vẽ điểm mốc lên ảnh
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)  

            # Hiển thị kết quả lên ảnh
            cv2.putText(frame, 'Pose Detected', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
    # Hiển thị webcam
    cv2.imshow('Webcam Pose Detection', frame)

    # Thoát khi nhấn phím 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import torch
import cv2
import numpy as np
import math
from ultralytics import YOLO # Import lop YOLO tu thu vien ultralytics

# --- 1. Ham Ho tro ---
def calculate_angle(p1, p2, p3):
    """
    Tinh toan goc (do) giua ba diem moc p1, p2, p3.
    p2 la dinh cua goc.
    """
    try:
        p1 = np.array(p1)
        p2 = np.array(p2)
        p3 = np.array(p3)

        # Tinh toan vector
        v1 = p1 - p2
        v2 = p3 - p2

        # Tinh toan Cosin goc
        cosine_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
        
        # Gioi han gia tri de tranh loi lam tron so trong arccos
        angle_rad = np.arccos(np.clip(cosine_angle, -1.0, 1.0))
        
        # Chuyen doi sang do
        angle_deg = np.degrees(angle_rad)
        return angle_deg
    except:
        return 0.0 # Tra ve 0 neu co loi (vi du: diem moc khong hop le)


# --- 2. Khoi tao Mo hinh va Webcam ---

# Tai mo hinh YOLO Pose
try:
    # Thay 'yolo11n-pose.pt' bang ten file mo hinh cua ban
    model = YOLO('yolo11n-pose.pt') 
except Exception as e:
    print(f"Loi khi tai mo hinh: {e}")
    print("Dam bao file 'yolo11n-pose.pt' ton tai va thu vien ultralytics da duoc cai dat dung cach.")
    exit()

# Khoi tao webcam
cap = cv2.VideoCapture(0)

# Nguong phan loai (Can dieu chinh dua tren thu nghiem va do phan giai webcam)
# Nguong lech X giua Tai va Hong (Don vi: pixel). Neu lon hon, coi la cui ve phia truoc.
BAD_POSTURE_THRESHOLD_X = 30 
# Nguong goc o lung/hong (Vai-Hong-Dau goi). 
# Neu goc nay < ANGLE_THRESHOLD_HIP, coi la gap hong qua muc.
ANGLE_THRESHOLD_HIP = 105 

# Chi so keypoints COCO
RIGHT_EAR = 4
RIGHT_SHOULDER = 6
RIGHT_HIP = 12
RIGHT_KNEE = 14
KEYPOINTS_COUNT = 17 # Tong so keypoints

# --- 3. Vong lap Xu ly Webcam ---
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Khong the doc frame tu webcam.")
        break

    # Du doan tu the (su dung device='cpu' de dam bao tinh tuong thich)
    results = model.predict(source=frame, device='cpu', verbose=False)
    
    posture_status = "Dang cho phat hien..."
    text_color = (255, 255, 255) # Trang

    if results and results[0].keypoints is not None:
        r = results[0]
        # Lay toa do keypoints (toa do pixel tho)
        keypoints_tensor = r.keypoints.xy 
        
        if keypoints_tensor.shape[0] > 0: # Neu phat hien it nhat mot nguoi
            keypoints = keypoints_tensor[0].cpu().numpy() # Lay keypoints cua nguoi dau tien
            
            # Kiem tra du keypoints quan trong de phan tich tu the
            if keypoints.shape[0] >= KEYPOINTS_COUNT and \
               keypoints[RIGHT_EAR].sum() > 0 and \
               keypoints[RIGHT_SHOULDER].sum() > 0 and \
               keypoints[RIGHT_HIP].sum() > 0 and \
               keypoints[RIGHT_KNEE].sum() > 0:
                
                # Lay toa do
                ear_point = keypoints[RIGHT_EAR]
                shoulder_point = keypoints[RIGHT_SHOULDER]
                hip_point = keypoints[RIGHT_HIP]
                knee_point = keypoints[RIGHT_KNEE]

                # --- PHAN LOAI TU THE ---
                
                is_bad_posture = False
                reasons = []

                # 1. Kiem tra do thang cua Cot song (Tai - Hong)
                # Tinh do lech X giua Tai va Hong. Do lech lon -> cui gap
                x_offset = abs(ear_point[0] - hip_point[0])
                
                if x_offset > BAD_POSTURE_THRESHOLD_X:
                    is_bad_posture = True
                    reasons.append(f"Cui gap/Lung cong ({x_offset:.1f} px)")
                
                # 2. Kiem tra goc o Hong (Vai - Hong - Dau goi)
                angle_hip = calculate_angle(shoulder_point, hip_point, knee_point)
                
                if angle_hip < ANGLE_THRESHOLD_HIP and angle_hip > 0: # angle_hip > 0 de tranh truong hop diem moc bi che
                    is_bad_posture = True
                    reasons.append(f"Ngoi gap hong ({angle_hip:.1f} do)")

                # --- CAP NHAT TRANG THAI ---
                
                if is_bad_posture:
                    posture_status = "Tu the SAI ❌"
                    text_color = (0, 0, 255) # Do
                    cv2.putText(frame, ", ".join(reasons), (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                else:
                    posture_status = "Tu the DUNG ✅"
                    text_color = (0, 255, 0) # Xanh
                
                # --- VE LEN ANH (Da sua loi TypeError) ---
                
                # Ve keypoints va duong noi (YOLO's plot function)
                r.plot(frame, labels=False, kpt_line=True, boxes=False) 
                
            else:
                posture_status = "Chua phat hien du diem moc"
                text_color = (0, 165, 255) # Cam

    # Hien thi trang thai phan loai
    cv2.putText(frame, posture_status, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, text_color, 2)
    cv2.imshow('Webcam Pose Detection', frame)

    # Thoat khi nhan phim 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [29]:
import torch
import cv2
import numpy as np
import os
import joblib
from ultralytics import YOLO 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# --- Cài đặt ---
MODEL_POSE_PATH = 'yolo11n-pose.pt'
DATA_DIR = './' # Thư mục chứa false_pose/ và true_pose/
OUTPUT_MODEL_PATH = 'posture_classifier_model.pkl'

# Tải mô hình YOLO Pose
try:
    model_pose = YOLO(MODEL_POSE_PATH) 
except Exception as e:
    print(f"Lỗi: Không thể tải mô hình YOLO Pose từ '{MODEL_POSE_PATH}'. Vui lòng kiểm tra file.")
    exit()

# --- A. Bước Trích xuất Keypoints ---
print("--- 1. BẮT ĐẦU TRÍCH XUẤT KEYPOINTS TỪ DỮ LIỆU ------------------")
output_data = []

# Duyệt qua các thư mục nhãn
for label_name in ['true_pose', 'false_pose']:
    label = 0 if label_name == 'true_pose' else 1 # 0: Đúng (True), 1: Sai (False)
    folder_path = os.path.join(DATA_DIR, label_name)
    
    if not os.path.isdir(folder_path):
        print(f"Lỗi: Không tìm thấy thư mục '{folder_path}'.")
        continue

    file_count = 0
    # Duyệt qua từng file ảnh trong thư mục
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.jpg', '.png', '.jpeg')):
            image_path = os.path.join(folder_path, filename)
            
            # Đọc ảnh
            img = cv2.imread(image_path)
            if img is None:
                continue
            
            # Dự đoán keypoints
            results = model_pose.predict(img, verbose=False)
            
            if results and results[0].keypoints is not None and results[0].keypoints.xy.shape[0] > 0:
                # Lấy keypoints chuẩn hóa (tọa độ 0-1) của người đầu tiên
                # Flatten để tạo thành 1 vector đặc trưng duy nhất: (x1, y1, x2, y2, ...)
                keypoints_norm = results[0].keypoints.xyn[0].cpu().numpy().flatten()
                
                output_data.append({
                    'features': keypoints_norm,
                    'label': label
                })
                file_count += 1
    print(f"Đã trích xuất {file_count} ảnh từ thư mục '{label_name}'.")

if not output_data:
    print("LỖI: Không trích xuất được keypoints nào. Vui lòng kiểm tra lại đường dẫn ảnh và model YOLO.")
    exit()

# Chuyển đổi sang mảng NumPy
X = np.array([item['features'] for item in output_data])
y = np.array([item['label'] for item in output_data])

print(f"\nHoàn tất trích xuất. Tổng số mẫu: {len(y)}")
print(f"Hình dạng dữ liệu đặc trưng (Features shape): {X.shape} (ví dụ: {len(y)} mẫu, {X.shape[1]} đặc trưng)")


# --- B. Bước Huấn luyện Mô hình Phân loại ---
print("\n--- 2. BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH PHÂN LOẠI RANDOM FOREST ------")

# Chia dữ liệu thành tập huấn luyện (80%) và tập kiểm tra (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Khởi tạo và huấn luyện mô hình Random Forest
# Random Forest là lựa chọn tốt vì nó không yêu cầu chuẩn hóa phức tạp và khó bị overfitting.
classifier = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
classifier.fit(X_train, y_train)

# Đánh giá mô hình
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Độ chính xác trên tập kiểm tra: {accuracy:.4f}")
print("Báo cáo phân loại:\n", classification_report(y_test, y_pred, target_names=['True Pose (0)', 'False Pose (1)']))

# --- C. Bước Lưu Mô hình ---
joblib.dump(classifier, OUTPUT_MODEL_PATH)
print(f"\n--- 3. HOÀN TẤT: Đã lưu mô hình phân loại: '{OUTPUT_MODEL_PATH}' ---")

--- 1. BẮT ĐẦU TRÍCH XUẤT KEYPOINTS TỪ DỮ LIỆU ------------------
Đã trích xuất 108 ảnh từ thư mục 'true_pose'.
Đã trích xuất 107 ảnh từ thư mục 'false_pose'.

Hoàn tất trích xuất. Tổng số mẫu: 215
Hình dạng dữ liệu đặc trưng (Features shape): (215, 34) (ví dụ: 215 mẫu, 34 đặc trưng)

--- 2. BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH PHÂN LOẠI RANDOM FOREST ------
Độ chính xác trên tập kiểm tra: 0.9535
Báo cáo phân loại:
                 precision    recall  f1-score   support

 True Pose (0)       0.94      0.94      0.94        16
False Pose (1)       0.96      0.96      0.96        27

      accuracy                           0.95        43
     macro avg       0.95      0.95      0.95        43
  weighted avg       0.95      0.95      0.95        43


--- 3. HOÀN TẤT: Đã lưu mô hình phân loại: 'posture_classifier_model.pkl' ---


In [4]:
import torch
import cv2
import numpy as np
import joblib
from ultralytics import YOLO 
from collections import deque
import time

# --- CÀI ĐẶT ---
MODEL_POSE_PATH = 'yolo11n-pose.pt'
CLASSIFIER_MODEL_PATH = 'posture_classifier_model.pkl'

# Tải mô hình YOLO Pose
try:
    model_pose = YOLO(MODEL_POSE_PATH) 
except:
    print(f"Lỗi: Không thể tải mô hình YOLO Pose từ '{MODEL_POSE_PATH}'. Vui lòng kiểm tra file.")
    exit()

# Tải mô hình phân loại đã huấn luyện
try:
    classifier = joblib.load(CLASSIFIER_MODEL_PATH)
except:
    print(f"Lỗi: Không thể tải mô hình phân loại. Vui lòng chạy bước huấn luyện trước.")
    exit()

# --- CẤU HÌNH LÀM MỊN VÀ ỔN ĐỊNH ---
SMOOTHING_WINDOW_SIZE = 5 
prediction_history = deque(maxlen=SMOOTHING_WINDOW_SIZE)
MIN_HOLD_FRAMES = 30 # ~ 1 giây (Ngưỡng giữ thông báo sau khi có thay đổi)

# --- BIẾN KIỂM SOÁT TRẠNG THÁI VÀ THỜI GIAN ---

# Trạng thái hiện tại đang được hiển thị
current_status = None 
hold_counter = 0      # Bộ đếm frame để ổn định thông báo

# Biến mới để theo dõi TỔNG THỜI GIAN NGỒI trong khung hình
total_sitting_time = 0.0 
prev_time = time.time() # Timestamp của frame trước

# Khởi tạo webcam
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    # Tính toán Delta Time (thời gian giữa 2 frame)
    current_frame_time = time.time()
    dt = current_frame_time - prev_time
    prev_time = current_frame_time

    results = model_pose.predict(source=frame, device='cpu', verbose=False)
    
    latest_prediction = -1 # -1: Không phát hiện
    person_detected = False

    if results and results[0].keypoints is not None and results[0].keypoints.xy.shape[0] > 0:
        r = results[0]
        person_detected = True
        
        # Trích xuất keypoints và dự đoán
        keypoints_norm = r.keypoints.xyn[0].cpu().numpy().flatten()
        features_input = keypoints_norm.reshape(1, -1) 
        latest_prediction = classifier.predict(features_input)[0]
        
        # Hiển thị keypoint
        r.plot(frame, labels=False, kpt_line=True, boxes=False) 

    # --- LÀM MỊN TEMPORAL VÀ ỔN ĐỊNH DỰ ĐOÁN ---
    
    if latest_prediction != -1:
        prediction_history.append(latest_prediction)
    
    mode_prediction = -1 
    if len(prediction_history) == SMOOTHING_WINDOW_SIZE:
        mode_prediction = max(set(prediction_history), key=prediction_history.count)
    elif latest_prediction != -1:
        mode_prediction = latest_prediction

    # --- CẬP NHẬT TRẠNG THÁI ỔN ĐỊNH VÀ BỘ ĐẾM THỜI GIAN ---
    
    should_change_status = False
    
    if person_detected:
        # Tăng tổng thời gian ngồi trong khung hình
        total_sitting_time += dt 
        
        # 1. Khởi tạo trạng thái đầu tiên
        if current_status is None:
            should_change_status = True
            
        # 2. Phát hiện thay đổi trạng thái (Sai <-> Đúng)
        elif mode_prediction != -1 and mode_prediction != current_status:
            if hold_counter >= MIN_HOLD_FRAMES:
                should_change_status = True
            else:
                hold_counter += 1
        
        # 3. Trạng thái ổn định trùng với trạng thái hiện tại (hoặc chưa đủ mẫu để làm mịn)
        elif mode_prediction != -1 and mode_prediction == current_status:
            hold_counter = 0 
            
    else: # person_detected == False: Người rời khỏi khung hình
        if current_status is not None:
            # RESET BỘ ĐẾM THỜI GIAN VÀ TRẠNG THÁI
            total_sitting_time = 0.0 
            current_status = None
            hold_counter = 0
    
    # Thực hiện thay đổi trạng thái (chỉ thay đổi current_status nếu cần)
    if should_change_status and mode_prediction != -1:
        current_status = mode_prediction
        hold_counter = 0

    # --- HIỂN THỊ KẾT QUẢ CUỐI CÙNG ---
    
    status_text = "Dang cho phat hien hoac on dinh..."
    time_text = ""
    text_color = (255, 255, 255)

    if current_status == 0:
        status_text = "Tu the DUNG ✅"
        text_color = (0, 255, 0) # Xanh
        time_text = f"Tong thoi gian ngoi: {total_sitting_time:.1f} giay"
    elif current_status == 1:
        status_text = "Tu the SAI ❌"
        text_color = (0, 0, 255) # Đỏ
        time_text = f"Tong thoi gian ngoi: {total_sitting_time:.1f} giay"
    elif not person_detected and current_status is None:
        status_text = "Khong phat hien nguoi 😴"
        
    cv2.putText(frame, status_text, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, text_color, 2)
    
    # Hiển thị thời gian ở dòng dưới
    cv2.putText(frame, time_text, (50, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, text_color, 2)
    
    cv2.imshow('Webcam Pose Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()